# Dimensionality reduction, and how to validate it

**Accompanies Section 7 of** *Best Practices for Unsupervised Learning in Molecular Systems* (Article v1.0).

Two demonstrations sit at the heart of this notebook: first, that unscaled features produce an embedding that looks *better* by conventional criteria while encoding nothing of chemical interest, and second, that neighbor embeddings manufacture convincing structure from data that contains none.

### Learning objectives
- Reproduce the scaled-versus-unscaled PCA comparison on QM7 Coulomb matrices
- See why explained variance is a trap
- Watch t-SNE fabricate clusters from pure noise, and put a number on it
- Report trustworthiness, continuity and neighborhood preservation, against a baseline

### What this notebook is designed to make go wrong
An embedding of random noise that yields a silhouette score most papers would describe as evidence of cluster structure.

### What you need installed
NumPy, pandas, scikit-learn and matplotlib, plus ASE, which reads the QM7 structure file.

### Roughly how long it takes
A few minutes. The seed-repeat loop in section 5 refits t-SNE five times and is the slow part.

In [ ]:
# CANONICAL SETUP CELL. Paste verbatim as the first code cell of every notebook.
# Import lines for libraries a given notebook does not use may be dropped, and
# lines it needs extra (pandas, rdkit, sklearn) may be added, but the style
# block, SEED and DATA must be identical everywhere.
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# One palette for every figure here. The series stay distinguishable in
# grayscale as well as in color, and marker and dash vary alongside the color,
# so nothing depends on color alone.
TEAL, PURPLE, LAVENDER, GREEN, PLUM, SLATE = (
    "#2D4F54", "#7B539E", "#B8A0D2", "#5A9448", "#9E4A78", "#3A3D4A"
)
PALETTE = [TEAL, PURPLE, LAVENDER, GREEN]


def set_style():
    """Apply the plot style used throughout these notebooks."""
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "axes.edgecolor": SLATE,
        "axes.labelcolor": SLATE,
        "axes.titlecolor": SLATE,
        "axes.linewidth": 1.0,
        "axes.grid": False,
        "xtick.color": SLATE,
        "ytick.color": SLATE,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "legend.frameon": False,
        "axes.prop_cycle": (
            mpl.cycler(color=PALETTE)
            + mpl.cycler(marker=["o", "s", "^", "D"])
            + mpl.cycler(linestyle=["-", (0, (4, 1.5)), (0, (1, 1.2)), (0, (5, 1.2, 1, 1.2))])
        ),
    })


set_style()
warnings.filterwarnings("ignore", category=FutureWarning)

# Fix a seed so the notebook reproduces. That is not the same as checking a
# conclusion survives a different seed, which we do explicitly where it matters.
SEED = 20260726
rng = np.random.default_rng(SEED)

# The example data, fetched by scripts/download_data.py.
DATA = Path.cwd().parent / "data"

## 1. Build Coulomb matrices for a fixed-size subset

We restrict to molecules with exactly 15 atoms. Zero-padding to a common size would otherwise
introduce a size artifact into every distance computation: a representation problem, not an
algorithm problem.

`qm7.xyz` is an extended-XYZ file, so ASE reads the whole thing in one call and hands
back a list of `Atoms` objects with the key=value pairs from each comment line in
`atoms.info`. The positions in this file are in bohr and not angstrom. That changes nothing
below, because every off-diagonal Coulomb entry is scaled by the same constant, but multiply
by 0.529177210903 before you quote a bond length or set a cutoff radius.

In [ ]:
from ase.io import read

frames = read(str(DATA / "qm7.xyz"), index=":")
sizes = np.array([len(atoms) for atoms in frames])
print(f"{len(frames)} QM7 structures, {sizes.min()} to {sizes.max()} atoms each")
print("first structure: "
      f"{len(frames[0])} atoms, atomization energy "
      f"{frames[0].info['atomization_energy']:.2f} kcal/mol")

subset = [atoms for atoms in frames if len(atoms) == 15]
print(f"\n{len(subset)} molecules with exactly 15 atoms")


def coulomb_matrix(positions, numbers):
    n = len(numbers)
    cm = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i == j:
                cm[i, j] = 0.5 * numbers[i] ** 2.4
            else:
                d = np.linalg.norm(positions[i] - positions[j])
                cm[i, j] = numbers[i] * numbers[j] / d
    return cm


lower = np.tril_indices(15, k=-1)
X = np.array([
    coulomb_matrix(atoms.get_positions(), atoms.get_atomic_numbers())[lower]
    for atoms in subset
])
heaviest = np.array([atoms.get_atomic_numbers().max() for atoms in subset])
energies = np.array([atoms.info["atomization_energy"] for atoms in subset])
print(f"feature matrix: {X.shape}")

### Name the ordering problem the fixed atom count hides

Restricting to molecules of one atom count makes the Coulomb vectors the same length, which is necessary but not sufficient: it does not make them well defined. The matrix is built from a list of atoms, and the same molecule written with its atoms in a different order gives a different matrix and so a different point in feature space. The rest of this notebook does not depend on the repair, and uses the raw off-diagonal vectors throughout, but the problem is worth seeing once. The two standard responses are to impose a canonical order by sorting rows and columns by their norm, and to take the sorted eigenvalues of the matrix, which are permutation-invariant by construction and far shorter. The eigenvalue form is not free: it discards which atom was which, so two genuinely different molecules can approach the same spectrum.

In [ ]:
# The Coulomb matrix depends on the order the atoms are listed in, so the same
# molecule with its atoms permuted lands at a different point in feature space.
atoms = subset[0]
pos, num = atoms.get_positions(), atoms.get_atomic_numbers()
perm = np.random.default_rng(SEED).permutation(len(num))  # local rng: do not disturb the shared stream
v_original = coulomb_matrix(pos, num)[lower]
v_permuted = coulomb_matrix(pos[perm], num[perm])[lower]
print(f"same molecule, atoms reordered: the off-diagonal vectors differ by "
      f"{np.linalg.norm(v_original - v_permuted):.1f} in L2 (identical would be 0.0)")


def sorted_coulomb(pos, num):
    """Coulomb matrix with rows and columns ordered by descending row norm."""
    M = coulomb_matrix(pos, num)
    order = np.argsort(-np.linalg.norm(M, axis=1))
    return M[np.ix_(order, order)]


s1 = sorted_coulomb(pos, num)[lower]
s2 = sorted_coulomb(pos[perm], num[perm])[lower]
print(f"after sorting rows and columns by norm: difference "
      f"{np.linalg.norm(s1 - s2):.3g} (a canonical order, invariant to the reordering)")

eig1 = np.sort(np.linalg.eigvalsh(coulomb_matrix(pos, num)))
eig2 = np.sort(np.linalg.eigvalsh(coulomb_matrix(pos[perm], num[perm])))
print(f"sorted eigenvalues: {len(eig1)} numbers instead of {len(lower[0])}, "
      f"difference {np.linalg.norm(eig1 - eig2):.3g} (invariant by construction)")

## 2. Scaling changes the answer

The Coulomb matrix diagonal scales as $0.5 Z^{2.4}$, so heavy atoms dominate every unscaled
distance. Watch what the first principal component encodes in each case.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pca_raw = PCA(n_components=30, random_state=SEED).fit(X)
X_scaled = StandardScaler().fit_transform(X)
pca_scaled = PCA(n_components=30, random_state=SEED).fit(X_scaled)

emb_raw = pca_raw.transform(X)
emb_scaled = pca_scaled.transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, emb, title in [(axes[0], emb_raw, "unscaled"), (axes[1], emb_scaled, "standardized")]:
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=heaviest, s=8, cmap="viridis", linewidths=0)
    ax.set_title(f"PCA of {title} Coulomb matrices")
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
plt.colorbar(sc, ax=axes, label="atomic number of heaviest atom", fraction=0.03)
plt.show()

corr_raw = np.corrcoef(emb_raw[:, 0], heaviest)[0, 1]
corr_scaled = np.corrcoef(emb_scaled[:, 0], heaviest)[0, 1]
print(f"|correlation of PC1 with heaviest atom|, unscaled     : {abs(corr_raw):.3f}")
print(f"|correlation of PC1 with heaviest atom|, standardized : {abs(corr_scaled):.3f}")

### Watch explained variance prefer the wrong answer

The unscaled analysis reaches 90% explained variance in far fewer components, which is not a
sign that it is better. Explained variance measures how *compactly* variance is captured, not
whether that variance means anything, and here a single dominant artifact concentrates
variance very efficiently.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.2))
for pca, label in [(pca_raw, "unscaled"), (pca_scaled, "standardized")]:
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    ax.plot(np.arange(1, len(cumulative) + 1), cumulative, "o-", ms=4, label=label)
    idx = int(np.searchsorted(cumulative, 0.9))
    if idx < len(cumulative):
        print(f"{label:14s}: {idx + 1} components reach 90% explained variance")
    else:
        print(f"{label:14s}: 90% not reached within {len(cumulative)} components "
              f"(cumulative {cumulative[-1]:.2f})")
ax.axhline(0.9, color="gray", ls="--", lw=1)
ax.set_xlabel("number of principal components")
ax.set_ylabel("cumulative explained variance")
ax.legend(frameon=False)
plt.show()

In [ ]:
# --- Numbers the article quotes -------------------------------------------
# These assertions exist because continuous integration proves the notebook
# *runs*; it does not prove it still says what the article says it says. A
# library default changes, a result shifts, CI stays green, and the article is
# quietly wrong. Tolerance bands, not equality: catch a change that matters,
# not floating-point noise. See Section 11.3 on dependency rot.

from sklearn.metrics import roc_auc_score

std_ratio = X.std(axis=0).max() / X.std(axis=0).min()
auc_raw = roc_auc_score(heaviest == 16, emb_raw[:, 0])
auc_scaled = roc_auc_score(heaviest == 16, emb_scaled[:, 0])
auc_raw, auc_scaled = max(auc_raw, 1 - auc_raw), max(auc_scaled, 1 - auc_scaled)
cum_raw = np.cumsum(pca_raw.explained_variance_ratio_)
cum_scaled = np.cumsum(pca_scaled.explained_variance_ratio_)
n90_raw = int(np.searchsorted(cum_raw, 0.90) + 1)

print(f"feature std ratio (max/min)        {std_ratio:8.0f}   article: 188")
print(f"|PC1 vs heaviest|, unscaled        {abs(corr_raw):8.3f}   article: 0.38")
print(f"|PC1 vs heaviest|, standardized    {abs(corr_scaled):8.3f}   article: 0.07")
print(f"AUC sulfur vs rest, unscaled       {auc_raw:8.2f}   article: 0.89")
print(f"AUC sulfur vs rest, standardized   {auc_scaled:8.2f}   article: 0.54")
print(f"components to 90% variance, raw    {n90_raw:8d}   article: 12")
print(f"cumulative variance at 30, scaled  {cum_scaled[-1]:8.2f}   article: 0.85")

assert 150 < std_ratio < 230, f"feature std ratio drifted: {std_ratio:.0f}"
assert 0.30 < abs(corr_raw) < 0.46, f"unscaled PC1 correlation drifted: {abs(corr_raw):.3f}"
assert abs(corr_scaled) < 0.15, f"standardized PC1 correlation drifted: {abs(corr_scaled):.3f}"
assert 0.84 < auc_raw < 0.94, f"unscaled AUC drifted: {auc_raw:.3f}"
assert auc_scaled < 0.62, f"standardized AUC drifted: {auc_scaled:.3f}"
assert 9 <= n90_raw <= 15, f"components to 90% drifted: {n90_raw}"
assert cum_scaled[-1] < 0.90, "standardized data now reaches 90% within 30 components"
print("\nall within tolerance of the values printed in the article")

### Compare t-SNE against PCA on the same descriptors

An earlier draft carried a figure here: $t$-SNE of the *same* standardized Coulomb
descriptors that Figure 3 shows under PCA, colored by atomization energy. It was cut for
space, and the point it made still holds, so it lives here instead.

There are three things to look for below. First, **the two methods group the same molecules
differently**: same input, same standardization, same random seed, different answer. Neither
is wrong; they optimize different things, and that is what "the embedding is part of the
model" means. Second, **the islands in the $t$-SNE panel sit at the scale set by the
perplexity, not at any scale intrinsic to the data**, which is why the next section embeds
pure noise and gets islands out of it. Perplexity is loosely how many neighbors each point is
asked to keep faith with: small values fragment the data into many small islands, large values
merge everything. Report the value you used, and treat a conclusion that survives only one
perplexity as no conclusion. Third, **the picture is a way to generate hypotheses and not
evidence that the groups exist**: read as a suggestion of where to look it is useful, read as
proof that the apparent islands are real chemical families it is inadmissible, for the reasons
the pure-noise example in the next section makes concrete.

The cell further down redraws this same plot at publication quality. The article
does not use it, so it is drawn inline for reference and not written to `figures/`.

### Measure how much of the neighborhood structure survived

The two panels below are the same molecules arranged two ways, so the question is how much
the arrangements agree. Three functions answer it, and section 4 turns them on five methods
at once.

**Trustworthiness** asks whether an embedding invented neighbors: points drawn close together
that were far apart in the input space. Low trustworthiness means the picture shows groupings
that are not in the data. scikit-learn ships it, so import it and move on. **Continuity** asks
the reverse, whether the embedding lost neighbors that were close to begin with, and it is the
same function called with the two spaces swapped. **Neighbor preservation** is the most direct
of the three: the fraction of each point's k nearest neighbors that are still among its k
nearest neighbors after the reduction.

In [ ]:
from sklearn.manifold import trustworthiness
from sklearn.neighbors import NearestNeighbors


def continuity(X, embedding, n_neighbors=12):
    """Neighbors the embedding lost, measured as trustworthiness in reverse."""
    return float(trustworthiness(embedding, X, n_neighbors=n_neighbors))


def knn_indices(A, n_neighbors):
    """Each point's k nearest neighbors, with the point itself dropped."""
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(A)
    return nn.kneighbors(A, return_distance=False)[:, 1:]


def neighbor_preservation(X, embedding, n_neighbors=12):
    """Mean fraction of each point's k nearest neighbors that survive the embedding."""
    before, after = knn_indices(X, n_neighbors), knn_indices(embedding, n_neighbors)
    return float(np.mean([
        np.intersect1d(before[i], after[i]).size / n_neighbors
        for i in range(len(before))
    ]))

In [ ]:
from sklearn.manifold import TSNE

# Same input, same seed, two methods.
pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(X_scaled)
tsne_2d = TSNE(n_components=2, perplexity=30, init="pca",
               random_state=SEED, max_iter=1000).fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, emb, title in [(axes[0], pca_2d, "PCA"), (axes[1], tsne_2d, "t-SNE, perplexity 30")]:
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=energies, s=7, alpha=0.75, linewidths=0)
    ax.set_title(f"{title} of the same standardized descriptors")
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(sc, ax=axes, label="atomization energy / kcal mol$^{-1}$", fraction=0.03)
plt.show()

# How much do the two arrangements agree about who is near whom?
overlap = neighbor_preservation(pca_2d, tsne_2d, n_neighbors=12)
print(f"12-NN overlap between the PCA and t-SNE arrangements: {overlap:.3f}")
print("Same molecules, same features, same seed. The disagreement is the method.")

# The islands move when the perplexity moves, which is the tell that their
# scale is a parameter, not a property of the chemistry.
for perp in (5, 30, 80):
    emb = TSNE(n_components=2, perplexity=perp, init="pca",
               random_state=SEED, max_iter=1000).fit_transform(X_scaled)
    print(f"  perplexity {perp:3d}: 12-NN overlap with PCA = "
          f"{neighbor_preservation(pca_2d, emb, n_neighbors=12):.3f}")

## 3. Neighbor embeddings manufacture structure

This is the demonstration behind Figure 4 of the manuscript. The data below is pure isotropic
noise in high dimensions: **it contains no clusters at all**. We embed it and then do what a
typical workflow does: cluster the embedding and report the best silhouette.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

noise = rng.standard_normal((400, 300))   # no structure whatsoever

def best_silhouette(emb, kmax=8):
    return max(
        silhouette_score(emb, KMeans(k, n_init=10, random_state=SEED).fit_predict(emb))
        for k in range(2, kmax + 1)
    )

sils = {}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, perplexity in zip(axes, [2, 5, 30]):
    emb = TSNE(n_components=2, perplexity=perplexity, init="pca",
               random_state=SEED, max_iter=1000).fit_transform(noise)
    sils[perplexity] = best_silhouette(emb)
    ax.scatter(emb[:, 0], emb[:, 1], s=5, alpha=0.6, linewidths=0)
    ax.set_title(f"perplexity {perplexity}\nbest silhouette = {sils[perplexity]:.2f}")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("t-SNE of 400 points of pure noise in 300 dimensions", y=1.04)
plt.show()

worst = max(sils, key=sils.get)
print(f"None of these clusters exist. The highest apparent silhouette is "
      f"{sils[worst]:.2f} at perplexity {worst}.")
print("Anything above ~0.3 is routinely reported in the literature as evidence")
print("of meaningful cluster structure. Low perplexity fragments hardest.")

In [ ]:
# --- Numbers the article quotes -------------------------------------------
# These assertions exist because continuous integration proves the notebook
# *runs*; it does not prove it still says what the article says it says. A
# library default changes, a result shifts, CI stays green, and the article is
# quietly wrong. Tolerance bands, not equality: catch a change that matters,
# not floating-point noise. See Section 11.3 on dependency rot.

# The article emphasizes the low-perplexity headline, so guard the worst case
# (the highest apparent silhouette over the perplexities scanned), not one panel.
s = max(sils.values())
print(f"highest apparent silhouette on noise  {s:8.2f}   article: 0.39")
assert 0.30 <= s <= 0.50, f"noise silhouette drifted: {s:.3f}; Figure 4 quotes 0.39"
print("within tolerance")

## 4. Validate the embedding quantitatively

Judge an embedding by numbers instead of by the picture it makes. `evaluate_embeddings` adds a
random projection baseline automatically, and an embedding that cannot beat a random
projection is not preserving neighborhoods, whatever it looks like. PCA is the linear
reference next to it, since a nonlinear method that cannot beat PCA has not earned its extra
parameters.

Trustworthiness, continuity and neighborhood preservation all score the *neighborhood
graph*, which is what a neighbor embedding optimizes, so t-SNE starts these three with an
advantage closer to a tautology than to a finding. Passing `y=energies` adds
`property_retention`, which no method here optimizes: of what the 105 descriptors know
about the atomization energy, how much survives the reduction to two dimensions? One means
the embedding is as good as the full descriptors, zero means it is no better than
predicting the mean.

Two details keep this an honest check rather than a restatement of the neighbor metrics. The readout is a ridge model rather than a k-nearest-neighbor one, so what earns a high score is a property that varies smoothly across the map, not the same local neighbor structure the three graph metrics already reward. And where a method exposes an out-of-sample transform, the cell below also fits the map on a training split and scores the held-out points, which is the situation a downstream user actually faces; MDS and t-SNE optimize every point jointly and have no such transform, so they are left out of that second table.

These functions are defined in the next cell.

In [ ]:
import pandas as pd
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.linear_model import Ridge
from sklearn.random_projection import GaussianRandomProjection


def ridge_mae(features, y, random_state, alpha=1.0):
    """Cross-validated mean absolute error of a ridge regressor for y.

    Ridge is a deliberate choice over a k-NN regressor here. Trustworthiness,
    continuity and neighbor preservation already score the neighborhood graph
    with a neighbor rule, so measuring property_retention with a second neighbor
    rule would reward the same local structure twice and a neighbor embedding
    would come out ahead by construction rather than by merit. A ridge model
    reads the property off the embedding coordinates globally instead, which
    turns this column into an independent probe: it asks whether the property
    varies smoothly across the map, not whether close points stayed close.
    """
    predicted = cross_val_predict(
        Ridge(alpha=alpha), features, y,
        cv=KFold(5, shuffle=True, random_state=random_state))
    return float(np.mean(np.abs(predicted - y)))


def property_retention(X, embedding, y, random_state=0):
    """Of what a linear model can read from X about y, how much survives.

    A ridge regressor is cross-validated three times: in the full feature space,
    in the embedding, and against the trivial model that always predicts the
    mean of y. The score places the embedding on that scale, so 1.0 means the
    embedding supports y as well as the input features do and 0.0 means it
    supports it no better than the mean. Values below zero say the embedding
    scrambled the property; values above one say the discarded dimensions were
    noise the full-space model was suffering from.
    """
    floor = float(np.mean(np.abs(y - y.mean())))
    ceiling = ridge_mae(X, y, random_state)
    got = ridge_mae(embedding, y, random_state)
    return (floor - got) / (floor - ceiling)


def out_of_sample_retention(X, y, estimators, random_state=0, alpha=1.0):
    """Split-based property retention for methods with an out-of-sample map.

    The transductive property_retention above embeds every point at once, so the
    ridge model is read on the same points the map was fit to. Only methods that
    expose a transform can be held to the stricter standard a downstream user
    actually faces: fit the embedding on a training set, project unseen points
    into it, and predict their property from where they land. PCA, Isomap and the
    random projection have such a map; MDS and t-SNE place points by optimizing
    all of them jointly and have none, so they cannot be scored this way and are
    marked out separately.
    """
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.3, random_state=random_state)
    floor = float(np.mean(np.abs(y_te - y_tr.mean())))
    ceiling_model = Ridge(alpha=alpha).fit(X_tr, y_tr)
    ceiling = float(np.mean(np.abs(ceiling_model.predict(X_te) - y_te)))
    rows = []
    for name, est in estimators.items():
        emb_tr = est.fit_transform(X_tr)
        emb_te = est.transform(X_te)
        model = Ridge(alpha=alpha).fit(emb_tr, y_tr)
        got = float(np.mean(np.abs(model.predict(emb_te) - y_te)))
        rows.append({"method": name,
                     "property_retention_oos": (floor - got) / (floor - ceiling)})
    return pd.DataFrame(rows)


def evaluate_embeddings(X, embeddings, n_neighbors=12, random_state=0, y=None):
    """Score several embeddings side by side, with a random projection for scale.

    Keep the random projection. It costs nothing, it comes with distortion
    guarantees (Johnson-Lindenstrauss), and it separates how much of an
    embedding's apparent quality is the method from how much is the data being
    easy to embed. Pass y to add the property_retention column.
    """
    embeddings = dict(embeddings)
    embeddings["random projection"] = GaussianRandomProjection(
        n_components=2, random_state=random_state).fit_transform(X)

    rows = []
    for name, emb in embeddings.items():
        row = {
            "method": name,
            "trustworthiness": float(trustworthiness(X, emb, n_neighbors=n_neighbors)),
            "continuity": continuity(X, emb, n_neighbors),
            f"{n_neighbors}nn_preserved": neighbor_preservation(X, emb, n_neighbors),
        }
        if y is not None:
            row["property_retention"] = property_retention(
                X, emb, y, random_state=random_state)
        rows.append(row)
    return (pd.DataFrame(rows)
            .sort_values("trustworthiness", ascending=False)
            .reset_index(drop=True))

In [ ]:
from sklearn.manifold import MDS, Isomap

embeddings = {
    "PCA": PCA(n_components=2, random_state=SEED).fit_transform(X_scaled),
    "Isomap": Isomap(n_components=2, n_neighbors=12).fit_transform(X_scaled),
    "MDS": MDS(n_components=2, random_state=SEED, normalized_stress="auto",
               n_init=1).fit_transform(X_scaled),
    "t-SNE": TSNE(n_components=2, perplexity=30, init="pca",
                  random_state=SEED, max_iter=1000).fit_transform(X_scaled),
}
quality = evaluate_embeddings(X_scaled, embeddings, n_neighbors=12,
                              random_state=SEED, y=energies)
quality.round(3)

In [ ]:
# The table above is transductive: every point was embedded at once, so the
# ridge model is read on the same points the map was fit to. For the methods
# that expose an out-of-sample transform, score the honest downstream case as
# well -- fit the map on a training split, project the held-out points into it,
# and predict their energy from where they land.
oos_estimators = {
    "PCA": PCA(n_components=2, random_state=SEED),
    "Isomap": Isomap(n_components=2, n_neighbors=12),
    "random projection": GaussianRandomProjection(n_components=2, random_state=SEED),
}
oos = out_of_sample_retention(X_scaled, energies, oos_estimators, random_state=SEED)
print("out-of-sample energy retention (fit on train, project the held-out 30%):")
print(oos.set_index("method").round(3).to_string())
print("\nMDS and t-SNE optimize all points jointly and expose no out-of-sample")
print("map, so they cannot be scored this way and are omitted here.")

In [ ]:
# --- Numbers the article quotes -------------------------------------------
# These assertions exist because continuous integration proves the notebook
# *runs*; it does not prove it still says what the article says it says. A
# library default changes, a result shifts, CI stays green, and the article is
# quietly wrong. Tolerance bands, not equality: catch a change that matters,
# not floating-point noise. See Section 11.3 on dependency rot.

row = quality.set_index("method")
print(f"{'method':<20}{'trust':>8}{'12nn':>8}{'energy kept':>14}")
for name in ("random projection", "PCA", "Isomap", "t-SNE"):
    r = row.loc[name]
    print(f"{name:<20}{r['trustworthiness']:8.3f}{r['12nn_preserved']:8.3f}"
          f"{r['property_retention']:14.3f}")

oos_row = oos.set_index("method")["property_retention_oos"]

# Figure 5 of the article. t-SNE leads the neighbor-graph axes, as a neighbor
# embedding must. But property_retention is now read with a ridge model rather
# than a k-NN one, so it no longer rewards the same neighbor structure twice,
# and on that independent probe t-SNE's lead disappears: Isomap edges it, and no
# embedding recovers the energy the full descriptors carry.
assert row.loc["t-SNE", "trustworthiness"] > row.loc["PCA", "trustworthiness"] > \
    row.loc["random projection", "trustworthiness"], "panel (a) ordering changed"
assert row["property_retention"].max() < 1.0, (
    "an embedding now matches the full descriptors; Figure 5's claim that "
    "reduction costs property information no longer holds"
)
assert row.loc["Isomap", "property_retention"] >= row.loc["t-SNE", "property_retention"], (
    "t-SNE now leads the ridge property probe; the article's point that the "
    "neighbor-graph lead does not carry to an independent readout no longer holds"
)
assert 0.15 <= row.loc["t-SNE", "property_retention"] <= 0.32, (
    f"t-SNE energy retention drifted: {row.loc['t-SNE', 'property_retention']:.3f}; "
    "the article quotes 0.23 with a ridge readout"
)
assert 0.04 <= row.loc["PCA", "property_retention"] <= 0.18, (
    f"PCA energy retention drifted: {row.loc['PCA', 'property_retention']:.3f}; "
    "the article quotes 0.10"
)
# Out-of-sample: only PCA, Isomap and the random projection have a transform.
assert oos_row["Isomap"] > oos_row["PCA"] > 0, (
    "out-of-sample retention ordering changed; the article reports Isomap "
    "holding up out of sample where PCA falls off"
)
assert 0.15 <= oos_row["Isomap"] <= 0.33, (
    f"Isomap out-of-sample retention drifted: {oos_row['Isomap']:.3f}; article quotes 0.24"
)
print("\nwithin tolerance")

## 5. Does the conclusion survive changing the seed?

Setting a seed makes a result reproducible, but it does not make that result *stable*, and
those are two different obligations that both have to be met.

In [ ]:
seeds = [0, 1, 2, 3, 4]
trust = []
for seed in seeds:
    emb = TSNE(n_components=2, perplexity=30, init="pca",
               random_state=seed, max_iter=1000).fit_transform(X_scaled)
    trust.append(trustworthiness(X_scaled, emb, n_neighbors=12))

print(f"t-SNE trustworthiness across {len(seeds)} seeds: "
      f"{np.mean(trust):.3f} +/- {np.std(trust):.3f}")
print(f"  range: [{min(trust):.3f}, {max(trust):.3f}]")

## 6. Report the four decisions behind a collective variable, and read the one diagnostic

For trajectory data the question is usually kinetic (which states exist, how
long the system stays in each), and the criterion is therefore *kinetic*, not
geometric. TICA and its neural extensions target that criterion directly. Four
decisions govern whether the result means anything, and all four have to be
reported:

1. **The lag time $\tau$** selects which processes count as slow. Plot the
   implied timescales against $\tau$ and use the shortest lag at which they have
   leveled off.
2. **The featurization** changes the answer at least as much as the algorithm
   does, and it can be scored. Any feature set gives a *lower bound* on the true
   slow eigenvalues, so the set with the higher VAMP-2 score is closer to the
   truth, provided the score is cross-validated over trajectories: without that
   it can be inflated arbitrarily by adding features.
3. **The number of components retained.**
4. **The seed.**

Only the first has a diagnostic you can read off a plot, and the cells below
build it on a three-state Markov chain instead of a real trajectory, because
with a toy chain we know the answer: there is one slow process, and its
timescale is fixed.

**What to watch:** a good discretization gives implied timescales that are flat
from the shortest lag. A discretization that cuts across the slow barrier
instead of along it gives timescales that climb with $\tau$ and never settle.
That is the signature the article describes: a result about your coordinate,
not a reason to pick the largest lag that still runs.

In [ ]:
def simulate_chain(T, n_steps, rng):
    """Sample a discrete-state trajectory from a transition matrix."""
    state, out = 0, np.empty(n_steps, dtype=int)
    cdf = np.cumsum(T, axis=1)
    for i in range(n_steps):
        out[i] = state
        state = int(np.searchsorted(cdf[state], rng.random()))
    return out


def implied_timescales(traj, n_states, lags):
    """t_i(tau) = -tau / ln(lambda_i(tau)); constant in tau iff Markovian."""
    out = []
    for lag in lags:
        C = np.zeros((n_states, n_states))
        for a, b in zip(traj[:-lag], traj[lag:]):
            C[a, b] += 1
        C = C + C.T                                   # enforce detailed balance
        T = C / C.sum(axis=1, keepdims=True)
        lam2 = np.sort(np.abs(np.linalg.eigvals(T)))[::-1][1]
        out.append(-lag / np.log(lam2) if 0.0 < lam2 < 1.0 else np.nan)
    return np.array(out)


rng_cv = np.random.default_rng(SEED)
p_slow, p_fast = 0.0015, 0.25
# One slow barrier (0 <-> 1) and one fast exchange (1 <-> 2).
T = np.array([[1 - p_slow, p_slow, 0.0],
              [p_slow, 1 - p_slow - p_fast, p_fast],
              [0.0, p_fast, 1 - p_fast]])
traj = simulate_chain(T, 600_000, rng_cv)

lags = [1, 2, 5, 10, 25, 50, 100, 250, 500, 1000]
good = implied_timescales((traj > 0).astype(int), 2, lags)   # cut AT the slow barrier
bad = implied_timescales((traj > 1).astype(int), 2, lags)    # cut ACROSS it

print(f"{'lag':>6s} {'good split':>12s} {'bad split':>12s}")
for lag, g, b in zip(lags, good, bad):
    print(f"{lag:6d} {g:12.1f} {b:12.1f}")

spread = lambda v: float(np.std(v[-3:]) / np.mean(v[-3:]))
print(f"\nrelative spread over the last three lags:")
print(f"  good discretization {spread(good):.3f}   <- plateaued")
print(f"  bad  discretization {spread(bad):.3f}   <- still climbing")
print("\nThe bad curve has not converged at any lag (which we can afford).")

fig, ax = plt.subplots(figsize=(5.0, 3.2))
for values, label, style in [(good, "cut at the slow barrier", "-"),
                             (bad, "cut across it", "--")]:
    ax.plot(lags, values, style, marker="o", ms=3.5, label=label)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"lag $\tau$ (steps)")
ax.set_ylabel("implied timescale (steps)")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## Manuscript figures


- **This is Figure 4 of the manuscript** (`embedding_illusion`): PCA and three t-SNE perplexities on 400 points of pure noise, each carrying the silhouette an analyst would report.
- **This is Figure 5 of the manuscript** (`embedding_quality`): five embeddings of the 15-atom QM7 molecules scored on neighborhood preservation and on retained energy information.
- `tSNE`: t-SNE of the standardized QM7 Coulomb descriptors colored by atomization energy. The article does not use it, so it is drawn inline below for reference but **not saved** to `figures/`.


This cell produces those three figures from the code that generated the published
versions. `embedding_illusion` is the demonstration of section 3 at manuscript quality, and
`embedding_quality` is the five-method comparison of section 4 over the same 1219
fifteen-atom QM7 molecules and the same 105 standardized Coulomb descriptors, with the seed
spread that section 5 argues for. Every figure is written as **both a PDF and a PNG**.

No figure here carries a plot title, because a published figure carries its title in its
caption. The panel identifiers you still see are `ax.text` and not titles, and they say
*which* panel this is, not what it claims. The cell also sizes the type for the printed
column, so it changes the shared plot settings; the last line calls `set_style()` again to
put them back.

In [ ]:
# --- Manuscript figures: embedding_illusion and embedding_quality (a t-SNE
# --- scatter is drawn inline for reference but not saved)
# --- (Section 7; embedding_illusion is Figure 4 of the article).
#
# The figure code lives here rather than in a script, so that the notebook a
# reader follows and the figure the article prints cannot drift apart. The QM7
# panels reuse X_scaled and energies from section 1, so the published figures
# and the analysis above are the same molecules and the same descriptors.

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, Isomap, TSNE
from sklearn.metrics import silhouette_score
from sklearn.random_projection import GaussianRandomProjection


import os


def savefig(fig, stem, outdir=None):
    """Write a figure as both a PNG and a PDF to a local ``figures/`` directory.

    Set the FIGDIR environment variable, or pass outdir, to write somewhere else.
    """
    outdir = Path(outdir or os.environ.get("FIGDIR") or "figures")
    outdir.mkdir(parents=True, exist_ok=True)
    for fmt in ("png", "pdf"):
        fig.savefig(outdir / f"{stem}.{fmt}")
    return outdir


def add_panel_label(ax, label, x=-0.09, y=1.01, size=8.5):
    """Put a bold panel label outside the axes, left of the y-tick labels.

    A panel label is text and not a title, so it stays in the printed figure.
    """
    ax.text(x, y, label, transform=ax.transAxes, fontsize=size,
            fontweight="bold", va="bottom", ha="left", color=SLATE)


def panel_caption(ax, text, size=7.0, pad=1.012):
    """Say which panel this is, above the axes, as text and not as a title.

    These are two different things. A title states the figure's claim, and it
    belongs in the caption. A panel identifier says which variant a panel shows
    ("t-SNE, perplexity 30"), and a multi-panel figure is unreadable without it.
    """
    ax.text(0.5, pad, text, transform=ax.transAxes, ha="center", va="bottom",
            fontsize=size, color=SLATE)


def scatter_cmap():
    """Near-white to purple, with the palest quarter of the ramp removed.

    One hue and monotone in lightness, so it reads as a magnitude and never as
    a set of categories. The light end goes because a pale filled cell reads as
    "low" while a pale scattered point reads as "not there".
    """
    base = mpl.colors.LinearSegmentedColormap.from_list(
        "manuscript_purple", ["#F6F2FA", LAVENDER, PURPLE])
    return mpl.colors.LinearSegmentedColormap.from_list(
        "manuscript_purple_marks", base(np.linspace(0.30, 1.0, 256)))


# Type sized for the printed column rather than for the screen. Each figure is
# authored at exactly the width it is included at, so LaTeX never rescales it
# and a 7.5 pt label is 7.5 pt on the page. The widths are measured from the
# compiled document: \linewidth = 250.95 pt = 3.47 in for one column of the
# two-column layout, \textwidth = 520.40 pt = 7.20 in for both.
COL_W, FULL_W = 3.47, 7.20
PANEL_SIZE = 8.5
PANEL_XY = (-0.09, 1.01)
mpl.rcParams.update({
    "figure.dpi": 200,
    "font.size": 7.5,
    "axes.labelsize": 7.5,
    "axes.titlesize": 8,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
    "xtick.major.size": 2.8,
    "ytick.major.size": 2.8,
    "lines.linewidth": 1.3,
})


def null_data(n=800, d=50, seed=SEED):
    """Isotropic Gaussian noise: a cloud with no cluster structure whatsoever."""
    return np.random.default_rng(seed).standard_normal((n, d))


def apparent_silhouette(emb, kmax=8):
    """Best silhouette an analyst could report by scanning k on the embedding."""
    return max(
        silhouette_score(emb, KMeans(k, n_init=10, random_state=SEED).fit_predict(emb))
        for k in range(2, kmax + 1))


def fig_qm7_tsne():
    """t-SNE on the same molecules and the same scaled descriptors as the PCA."""
    emb = TSNE(n_components=2, perplexity=30, init="pca",
               random_state=SEED, max_iter=1000).fit_transform(X_scaled)

    fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.86))
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=energies, cmap=scatter_cmap(),
                    s=5, alpha=0.75, linewidths=0)
    cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label("atomization energy (kcal mol$^{-1}$)", fontsize=7)
    cb.ax.tick_params(labelsize=6.5)
    cb.outline.set_edgecolor(SLATE)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.set_xticks([])
    ax.set_yticks([])
    # The article does not use this scatter, so it is drawn inline for the
    # discussion above and not written to figures/.
    print("drew the t-SNE scatter inline (not saved)")


def fig_embedding_illusion():
    # High ambient dimension relative to sample size is the regime in which
    # neighbor-based embeddings shatter structureless data into islands, and it
    # is the regime of most molecular descriptor matrices.
    noise = null_data(n=400, d=300)

    perplexities = [2, 5, 30]
    fig, axes = plt.subplots(1, 4, figsize=(FULL_W, 2.75))
    fig.subplots_adjust(wspace=0.28)

    pcs = PCA(n_components=2, random_state=SEED).fit_transform(noise)
    axes[0].scatter(pcs[:, 0], pcs[:, 1], s=3.5, c=SLATE, alpha=0.5, linewidths=0)
    panel_caption(axes[0], "PCA")
    axes[0].set_xlabel("Component 1")
    axes[0].set_ylabel("Component 2")
    axes[0].text(0.5, -0.34, f"apparent $S$ = {apparent_silhouette(pcs):.2f}",
                 transform=axes[0].transAxes, ha="center", fontsize=6.5, color=PLUM,
                 weight="bold")
    axes[0].set_box_aspect(1)

    for ax, perp in zip(axes[1:], perplexities):
        emb = TSNE(n_components=2, perplexity=perp, init="pca",
                   random_state=SEED, max_iter=1000).fit_transform(noise)
        ax.scatter(emb[:, 0], emb[:, 1], s=3.5, c=SLATE, alpha=0.5, linewidths=0)
        panel_caption(ax, f"$t$-SNE, perplexity {perp}")
        ax.set_xlabel("Component 1")
        ax.set_yticklabels([])
        ax.set_box_aspect(1)
        ax.text(0.5, -0.34, f"apparent $S$ = {apparent_silhouette(emb):.2f}",
                transform=ax.transAxes, ha="center", fontsize=6.5, color=PLUM,
                weight="bold")

    # The t-SNE panels keep their x tick labels so every panel's x-axis label
    # sits at the same height and the four line up; the y-label is still dropped
    # on those panels, where it collided with the neighbor at this size.
    for ax, lab in zip(axes, ["(a)", "(b)", "(c)", "(d)"]):
        add_panel_label(ax, lab, x=PANEL_XY[0], y=PANEL_XY[1], size=PANEL_SIZE)
    outdir = savefig(fig, "embedding_illusion")
    print(f"wrote embedding_illusion.png and .pdf to {outdir}")


def fig_embedding_quality():
    """Figure 5: five embeddings of QM7, scored on two different questions.

    Panel (a) scores the neighborhood graph, which is what a neighbor embedding
    optimizes. Panel (b) scores something none of these methods optimizes: how
    much of what the descriptors know about the atomization energy survives the
    reduction. That is the panel that settles whether you can analyze chemistry
    in a two-dimensional picture.
    """
    methods = ("Random\nprojection", "PCA", "Isomap", "MDS", "t-SNE")
    stochastic = {"Random\nprojection", "MDS", "t-SNE"}
    out_of_sample = {"Random\nprojection", "PCA", "Isomap"}   # these expose a .transform
    k = 12
    seeds = (0, 1, 2, 3, 4)

    def make(name, seed):
        return {
            "Random\nprojection": GaussianRandomProjection(
                n_components=2, random_state=seed),
            "PCA": PCA(n_components=2, random_state=SEED),
            "Isomap": Isomap(n_components=2, n_neighbors=k),
            "MDS": MDS(n_components=2, random_state=seed,
                       normalized_stress="auto", n_init=1),
            "t-SNE": TSNE(n_components=2, perplexity=30, init="pca",
                          random_state=seed, max_iter=1000),
        }[name]

    means, sds = {}, {}
    for name in methods:
        rows = []
        for seed in (seeds if name in stochastic else (SEED,)):
            emb = make(name, seed).fit_transform(X_scaled)
            rows.append((float(trustworthiness(X_scaled, emb, n_neighbors=k)),
                         continuity(X_scaled, emb, k),
                         neighbor_preservation(X_scaled, emb, k),
                         property_retention(X_scaled, emb, energies,
                                            random_state=SEED)))
        a = np.asarray(rows)
        means[name], sds[name] = a.mean(0), a.std(0)

    labels = [f"{n}$^{{\\dagger}}$" if n in out_of_sample else n for n in methods]
    x = np.arange(len(methods))
    xlim = (-0.62, len(methods) - 0.4)

    fig, (axA, axB) = plt.subplots(
        1, 2, figsize=(FULL_W, 2.8),
        gridspec_kw={"width_ratios": [1.62, 1], "wspace": 0.26})
    fig.subplots_adjust(top=0.85, bottom=0.20)

    series = [("trustworthiness", 0, PALETTE[0], ""),
              ("continuity", 1, PALETTE[1], "///"),
              (f"{k}-NN retained", 2, PALETTE[2], "...")]
    width = 0.27
    for offset, (label, idx, color, hatch) in zip((-width, 0.0, width), series):
        axA.bar(x + offset, [means[n][idx] for n in methods], width,
                yerr=[sds[n][idx] for n in methods],
                error_kw=dict(elinewidth=0.7, capsize=1.6, ecolor=SLATE),
                label=label, color=color, hatch=hatch,
                edgecolor=SLATE, linewidth=0.6)
    axA.set_xticks(x)
    axA.set_xticklabels(labels)
    axA.set_ylabel("score (higher is better)")
    axA.set_ylim(0, 1.05)
    axA.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.06), fontsize=6.5)

    axB.bar(x, [means[n][3] for n in methods], 0.55,
            yerr=[sds[n][3] for n in methods],
            error_kw=dict(elinewidth=0.7, capsize=1.8, ecolor=SLATE),
            color=PALETTE[0], edgecolor=SLATE, linewidth=0.6)
    axB.axhline(0.0, ls=":", lw=0.9, color=SLATE)
    axB.set_xticks(x)
    axB.set_xticklabels(labels)
    axB.set_ylabel("energy information retained")
    axB.set_ylim(-0.05, 1.0)

    # The floor method, shaded in both panels so it reads as the baseline and
    # not as a fifth competitor.
    for ax in (axA, axB):
        ax.axvspan(xlim[0], 0.5, color=LAVENDER, alpha=0.35, lw=0, zorder=0)
        ax.set_xlim(*xlim)

    for ax, lab in ((axA, "(a)"), (axB, "(b)")):
        add_panel_label(ax, lab, x=PANEL_XY[0], y=1.05, size=PANEL_SIZE)

    outdir = savefig(fig, "embedding_quality")
    for name in methods:
        flat = name.replace("\n", " ")
        print(f"  {flat:<18} trust {means[name][0]:.3f}  cont {means[name][1]:.3f}  "
              f"{k}-NN {means[name][2]:.3f}  energy kept {means[name][3]:.3f}"
              f" +/- {sds[name][3]:.3f}")
    print(f"wrote embedding_quality.png and .pdf to {outdir}")


fig_qm7_tsne()
fig_embedding_illusion()
fig_embedding_quality()
plt.show()

# Put the shared plot settings back, so anything you add below this point gets
# screen-sized type again.
set_style()

### Exercise

Color the standardized PCA embedding by atomization energy (`energies`) instead of by heaviest
atom. Is the structure you see chemical, or is it still compositional? How would you tell?

In [ ]:
# YOUR CODE HERE